# Timeseries processing over a period

In [ ]:
from __future__ import annotations

In [ ]:
%load_ext autoreload
%autoreload 2

In [ ]:
import datetime as dt
import os
import shutil
from pprint import pprint

import matplotlib.pyplot as plt
import xarray as xr

In [ ]:
from sweat.api import read_ts_input_data, run_timeseries, run_window_time_series
from sweat.common.constant import ETVar
from sweat.common.io import read_data_from_file
from sweat.timeseries.config import (
    WindowTimeSeriesInputFile,
    check_config_window_timeseries,
)
from sweat.timeseries.timeseries_handler import create_config, window_generator

In [ ]:
import logging

from sweat.logging import LoggerManager

LoggerManager.set_level(logging.INFO)

In [ ]:
def locate_test_data() -> str:
    """
    Locate test data directory
    """
    current = os.getcwd()
    parent = os.path.dirname(current)
    if os.path.isdir(os.path.join(parent, "tests")):
        return os.path.join(parent, "tests", "data", "timeseries")
    parent = os.path.dirname(parent)
    if os.path.isdir(os.path.join(parent, "tests")):
        return os.path.join(parent, "tests", "data", "timeseries")
    msg = "Test data directory not found"
    raise FileNotFoundError(msg)


def clean(ts, config):
    """
    Clean outputs
    """
    # Remove directories
    shutil.rmtree(config, ignore_errors=True)
    shutil.rmtree(ts, ignore_errors=True)
    # Recreate directories
    os.makedirs(config, exist_ok=True)
    os.makedirs(ts, exist_ok=True)


def get_et_time_series(path_dir: str, date: str) -> xr.DataArray:
    """
    Return the simulated evapotranspiration for a given date
    from the corresponding GeoTIFF file in the specified directory.

    Parameters
    ----------
    dir: str
        Directory where the `et_time_series_YYYYMMDD.tif` files are stored
    date: str
        Date

    Returns
    -------
    xr.DataArray
        Simulated evapotranspiration spatial distribution
    """
    date_dt = dt.datetime.strptime(date, "%Y-%m-%d")  # noqa DTZ007
    filename = f"et_time_series_{date_dt.strftime('%Y%m%d')}.tif"
    path = os.path.join(path_dir, filename)
    if not os.path.isfile(path):
        msg = f"ET TS File not found for date {date}"
        raise ValueError(msg)
    data = read_data_from_file(path)
    return data[ETVar.ET.value]


def get_et_single_date(path_dir: str, date: dt.datetime) -> xr.DataArray:
    """
    Return the observed evapotranspiration for a given date
    from the corresponding GeoTIFF file in the specified directory.

    Parameters
    ----------
    dir: str
        Directory where the `et_single_date_YYYYMMDD.tif` are stored
    date: dt.datetime
        Date of the acquisition

    Returns
    -------
    xr.DataArray
        Observed evapotranspiration spatial distribution
    """
    date_dt = dt.datetime.strptime(date, "%Y-%m-%d")  # noqa DTZ007
    filename = f"et_single_date_{date_dt.strftime('%Y%m%d')}.tif"
    path = os.path.join(path_dir, filename)
    if not os.path.isfile(path):
        msg = f"ET File not found for date {date}"
        raise ValueError(msg)
    data = read_data_from_file(path)
    return data[ETVar.ET.value]


def plot(out_dir: str, ref_dir: str, date: str):
    """
    Plot ET corresponding to a date
    """
    plt.figure(figsize=(12, 5))
    plt.subplot(1, 2, 1)
    et_ref = get_et_single_date(ref_dir, date)
    et = get_et_time_series(out_dir, date)

    vmin = 0  # min(et.min(),et_ref.min())
    vmax = max(et.max(), et_ref.max())
    et_ref.plot.imshow(vmin=vmin, vmax=vmax)
    plt.title(f"Observed evapotranspiration {date}")

    plt.subplot(1, 2, 2)
    et.plot.imshow(vmin=0, vmax=vmax)
    plt.title(f"Simulated evapotranspiration {date}")
    plt.show()

In [ ]:
output = os.path.join("out", "timeseries")

# Case: Simple execution

Input configuration

In [ ]:
config_dict = {
    "input": {
        "period_start": "2025-08-23",
        "period_end": "2025-08-29",
        "et_single_date_dir": locate_test_data(),
        "radiation_dir": locate_test_data(),
    },
    "output": {
        "path": output,
    },
}

#### Execution

Check input configuration

In [ ]:
config_dict_checked = check_config_window_timeseries(config_dict)

In [ ]:
pprint(config_dict_checked)

In [ ]:
config = WindowTimeSeriesInputFile.model_validate(config_dict_checked)
run_window_time_series(
    period_start=config.input.period_start,
    period_end=config.input.period_end,
    et_single_date_dir=config.input.et_single_date_dir,
    radiation_dir=config.input.radiation_dir,
    et_time_series_dir=config.output.path,
    window=config.input.window,
    shift=config.input.shift,
    params=config.params,
    debug=config.debug.get_debug_config(),
    **config.debug.get_timeseries_config(),
)

Visualize results

In [ ]:
plot(output, locate_test_data(), date="2025-08-29")

## Case: Detailed configuration

Input configuration

In [ ]:
config_dict = {
    "input": {
        "period_end": "2025-08-29",
        "period_start": "2025-08-23",
        "shift": 2,
        "window": 4,
        "et_single_date_dir": locate_test_data(),
        "radiation_dir": locate_test_data(),
    },
    "output": {"path": "out/timeseries"},
    "params": {
        "stack": {"et_single_date_filtering": {}},
        "update": {
            "method": "linear",
            "params": {"radiation_mode": 0, "strict_mode": True},
        },
    },
    "debug": {
        "config_verbose": True,
    },
}

#### Execution

Check input configuration

In [ ]:
config_dict_checked = check_config_window_timeseries(config_dict)

In [ ]:
pprint(config_dict_checked)

In [ ]:
config = WindowTimeSeriesInputFile.model_validate(config_dict_checked)
run_window_time_series(
    period_start=config.input.period_start,
    period_end=config.input.period_end,
    et_single_date_dir=config.input.et_single_date_dir,
    radiation_dir=config.input.radiation_dir,
    et_time_series_dir=config.output.path,
    window=config.input.window,
    shift=config.input.shift,
    params=config.params,
    debug=config.debug.get_debug_config(),
    **config.debug.get_timeseries_config(),
)

In [ ]:
plot(output, locate_test_data(), date="2025-08-29")

## Case: detailed pipeline

Input

In [ ]:
min_date_str = "2025-08-23"
max_date_str = "2025-08-29"

In [ ]:
min_date_dt = dt.datetime.strptime(min_date_str, "%Y-%m-%d")  # noqa: DTZ007
max_date_dt = dt.datetime.strptime(max_date_str, "%Y-%m-%d")  # noqa: DTZ007

Input directories

In [ ]:
et_single_date_dir = locate_test_data()
radiation_dir = locate_test_data()

Output directories

In [ ]:
output_path = "out"
os.makedirs(output_path, exist_ok=True)

In [ ]:
et_time_series_dir = os.path.join(output_path, "et_timeseries")
config_dir = os.path.join(output_path, "config_dir")
clean(et_time_series_dir, config_dir)

Generating start and end dates for each sliding window shift within the given period (a 7-day window with a 1-day shift)

In [ ]:
for i in window_generator(min_date_dt, max_date_dt, 7, 1):
    print(i)

In [ ]:
windows = window_generator(min_date_dt, max_date_dt, 7, 1)

Selecting dates corresponding to the first window, that's to say the first date only

In [ ]:
window_start, window_end = next(windows)

In [ ]:
print(window_start, window_end)

Creating the configuration dictionary with the dates and file paths associated

In [ ]:
config = create_config(
    window_start=window_start,
    window_end=window_end,
    et_single_date_dir=et_single_date_dir,
    radiation_dir=radiation_dir,
    et_time_series_dir=et_time_series_dir,
    verbose=True,
    config_dir=config_dir,
)

In [ ]:
pprint(config.model_dump())

Running timeseries and creating the resulting et_time_series .tif file

In [ ]:
# Clean
clean(et_time_series_dir, config_dir)
logging.info("Clean output: OK")
# Read timeseries input
et_ts, radiation_ts, et_sd, dem = read_ts_input_data(config.input.model_dump())
logging.info("Read timeseries input data: OK")
# Run timeseries
updated_ts = run_timeseries(
    et_ts=et_ts,
    radiation_ts=radiation_ts,
    et_sd=et_sd,
    dem=dem,
    params=config.params.model_dump(),
    debug=config.debug.model_dump(),
)
logging.info("Update time series: OK")

Running the global function that performs the various operations previously described, for the entire given period

In [ ]:
# Clean output
clean(et_time_series_dir, config_dir)
# Run timeseries over a period
run_window_time_series(
    period_start=min_date_dt,
    period_end=max_date_dt,
    et_single_date_dir=et_single_date_dir,
    radiation_dir=radiation_dir,
    et_time_series_dir=et_time_series_dir,
    window=7,
    shift=1,
    config_verbose=True,
    config_dir=config_dir,
)

Visualize results

In [ ]:
plot(et_time_series_dir, et_single_date_dir, date="2025-08-29")